# 10 Mini Crosswords: Your Heuristic vs ToT-style DFS
Self-contained Kaggle notebook. Enable **GPU** and **Internet**, then run the cells in order. The benchmark is disabled until you set `RUN_BENCHMARK=True`.

**10 puzzles by default**, using the first ten entries of the fixed 50-puzzle pool from the [original ToT mini-crossword dataset](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/src/tot/data/crosswords/mini0505.json). Set `PUZZLE_LIMIT` before running to select a larger prefix. Both methods use identical puzzles and budgets. This is a custom seeded subset; selected IDs and source revision are saved.

Two methods: **ToT-style DFS** and **five-factor heuristic best-first**. The [official ToT crossword implementation](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/scripts/crosswords/search_crosswords-dfs.ipynb) uses DFS. Our baseline adapts it to a local model and a shared engine; it is not an exact reproduction or a claim of current SOTA.

Both methods receive the same model, prompts, revision rules and limits. They run one puzzle at a time, printing both results before proceeding. Reference solutions never enter model prompts, pruning, stopping or final-state selection.

**Not run on a Kaggle GPU during creation.** Offline tests use mocks. Other benchmarks and previous project files are unchanged.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers>=4.51,<5','accelerate>=1.6,<2','bitsandbytes>=0.45,<1',
    'huggingface_hub<1','lm-format-enforcer==0.11.3','pandas','matplotlib'])
# Restart the kernel after installation, then continue with the next cell.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, zipfile
PROJECT=Path('/kaggle/working/crosswords10_schema_v3')
PROJECT.mkdir(parents=True,exist_ok=True)
# Preserve the original selection and model pin when upgrading an existing session.
import shutil
previous=Path('/kaggle/working/crosswords50_schema_v2')
if not (previous/'data').exists():previous=Path('/kaggle/working/crosswords50')
if (previous/'data').exists() and not (PROJECT/'data').exists():
    shutil.copytree(previous/'data',PROJECT/'data')
if (previous/'model_pin.json').exists() and not (PROJECT/'model_pin.json').exists():
    shutil.copy2(previous/'model_pin.json',PROJECT/'model_pin.json')
payload=base64.b64decode("UEsDBBQAAAAIAC+YYV3pNIPuJB8AAJxeAAASAAAAY3Jvc3N3b3JkX2JlbmNoLnB5pTxrc+M2kt9Tlf/AcSpF0qZpyVnnduUgtbMZ53a2stm5mcldXRSViqYgixFFcvjwIzr99+sHAIIPeSZZV82IBIFGo9HoFxo4OTl5/5CfF3maxE/OLsmS87jMq+ohL1fOrczizS4qt6HzTkZlvHEyeS9Lp5SxTO5lBQ9rWUIl6URZ9SDLKjw5Ofn8s3WZ75xVVEdxGlUV1Et2RV7WbVHgRNUqiWtVtYjqTZrc6mpv4PXzz9RLXN2b501UYT3z/muVZ+ZlZ7cqo2yV79pXaR7rZNe+NGUK8MJSfmhkBdh8/tk/b97//V+v3jnC8dw6r5erdeUGjruRTZlUdRIvb6HiOimr2vU//+zVzfcvf/rhPdTG0Xi76HFZ51uZVWJ6OYG/wMGiOEpTKPlavcrHAqiV5FALij7/zOn+YZWVLOqNuIQGtzCSeLNcR3Gdl+IrhgBEy+pElqPt72Qmy6gG+BqXr64AkLyP0qZTfDUZaZ01u2VcP4o/T/9yGTi13BUIrCmlmIRfB05RNpkEGqT5AxRMrgKag2Up6zKRlbgcAnyQyd2mrgQR6EMTpUn9JMKvoGXVxLGsKngB9HjEBDHOq1qEU3haAZOVFTWYXvk+zs9KrqH4DibBy29/9WfcH/TflJnmj7DaRJdXX3uIWrhqdkWFdaFDmPLlVj5V4n3ZSD8Exs1X0vP9cCMfFVDTyUOZ1HKJIDxkz8BB+kndIRbBrCOn0mf/morCIoLlUIe77SopPX7h3mACHoGBlvmWO2cw9a4AKNTyIak3y6pZr5NHghjy85kbQiW3rR8yYrV8rO0BEnaBk2Qr6FJc+tAfji7J7oTb1OvzP7uAIjYvZZFGsWSs9WDXso43HqwGPT7Eprc6QnjNC5l5veK3/IvFgbOR0QqmTOzdnypZnr8EZqzdmWtEynkpK5Ik7gFQxLWYNzWwsQ8CAWaxKmBVyFnLRGpi9RfoNFq1c1TAYIDGS5Qrlay9Ms9rmGaZypgYvZJyBavo8ms9LJBOb5LMaYqqBkg7gHuf4Eq8dqpoV6TSuZo4TZbAgJy7MllVIAHXeYnSDWQj8ErqVHFeAlFZzhGG0KfmBHwGMuPPJ7PALsqSNdBvqVgKG1+4Vd6UsVzqjyHOtMsNknW3TUhQK8+3yLaTdQSwiD/SPFpVXrcJkpFZyPfbVggZGs7dLgXdhfNC9Kg6667zMkoq6fw38uBNWeal577T1R2s7sSbKLuDZk4D9SJnDRO6cdS0wXoGhQLi7Sl0LWzK6AGlMNMDFdPkanLFhPB5ALdPtaw8qwkqm1ZNaDEAgDor3BFCj5OpzPXcBQOSaY8DixzwcAuY9ljWeXaepcUFsI88z9fn9SZvQLydp+nObdvE+W6X1N0J4CW2djd1XVSzi4uoSMI7WGXNbQjVL7Cb6mKPP4cLbl9d7KKqlqXr+4DqJtIIqqnKctRr4bqBviOEXbrzyflfovP1Yv+nycENFBr+bGR6XmcgMJLVcCXYMwArGsbQogyEVCjDLJYx6CCUdYi9wnvPPR4uqjK+AO15gTN8YRY/DKgzjf25bqVQ+wXMEtBVkufa1PkjOP3w+rubH9/d2AM8wlwsYZm7kHkGDd7966e3390sFcSwfqx7rTpY+4N1SbqQZjxBthf4qKdL8E+A1BckUztcKj7C3APx1331tb6McT66HNoOVYlA4ewPoNea335LifZzxYAgEknRPAaOFwNPgTmXyho4tfKhHBRPs0OjQXqqG1sy3eYRGJawntzw1zzJPN0wbAqwNLyuNEpB21AH/gsxnTjQLZYQCCi5vMKSwTJw5y/Pf17sL69wCXDdj0mrtfvPKIVR7UBS8XCdPQ3w0FkORJUQJBbonqhJa8Yk4OlMVmJt6TluP5v8aYVoqCmkMqFIRwMTin5VnjY4SYIRVr3GYMUmsIaI+mmCSpZxIG1fGeGtSWWq+99cTcaW/Y8oi6MaaoNkQWW3QqM2i2uFoZ5sPex27tmgDt/Sj9djqpC1p4VAAMC1ggOeh9nFYk/RIc4bMFIQZc0hgaLuktSu6I0G1wa2uBoxWBW/y9USG86LuZugxkIeLZAb1RAWgVY3eiGpFaO++0ch46y4OEwgHVOhbyVcO3FT1VBeFSmuXGTJeoPEvIVJ20C79/l7MHdvoXfX/4gWB1tj9WmKG7UyEtdWVoMZf6V0bFHm9zKL0FVTulhjYhm6nb4CAj6sxCKwFepXE5abgSa1avKF9hmTrGhqJyrrBL0YMIKASnFSp0/wCOy/Av5i79FB2zw80iEZmeg0tf3N99vZPc30NrinuQ6h0Q6JCOTdvoBpU8vKPQwZwohCsjBVqbYsYwkum1elYM91XQxahSUSkL6eoh9Dv2fgoGC3+PbNFVkR/crnV8HlVWA5Mli4zKKdHOvJczduH6B77/pnoLGp/pdXZ1MDiuTGMl97D7bE1dJ27i7dxenlVSu/EUDgkJffEdkPfYGtLA0snzmoX5Oske1nSxewKEdwvyWFZ1GQ++lL4UQhPafmC+olwWEv3YBBjVou3+XZGvgH0LhDey+5y3ZoYduC2oxcgxYKtw59tQZiiWtcCnKOiAqBRSTLgXirLCUQ4MDfFdLEIESroodZgB1GYGJgcQpYZOCF30vl2cuqdSUUqQF6VtW4VIl8QZLVvtZ0k28E8QNrw2O6D1Vfn+Rq1D/mmTS90TDnCHCBRjHP8aAi2Oqg7AQxM7PHtSrjpsJVllwdlXeyZk+/zwKEjt/yXw4CsgycPF31TQYE3GNArIViEdwwj6YU23BvVM6ParZfCMWILWuOcSYh4AcA2h/YBzw4qtGOrjNkHE6Hm+oGFaDCHpnprybY9fln9OO8q2GA2r9GOs64FcWa3MA/nU74I8VCZoAuuhCq7MPMWYNCoJJQlRV2mVrbyI3JCkNyvQZaTDRAkZUOFyhDBkM5mgh1+WSR41EQEK7vD7hpFz16kzAAA9qbhsEjS0AMxIVJtQaWhwl99Fl0qa6Uj/UYwxgdr13Ywfungp98w4KmCeMOnvQKTERlMQ3lnOjJwO5q/yVT633t7lu5m/iHmbMnkPNkcXD+D4MxuEbF/uTEkg/zLZsUW2QiZqDE97G+Vmdiz6spWeDKPGlgGtJUrk4OLjMiNmRVMJ20GiAGrVxGqB+QOXoaoGp2XgxabMkgYgRhxkgNQh6pf4GiXYVykixKl9DTtgvzC+c1hoaSOEq1ujX2DYG/BX5Hlb/J0aIZRHZBgUuweqL7KEmj21SGXWXVG8gpo/ch6I8QhCp9KQLnnJ+I261IFEUbK64eODrUCYDWd4GDNgDwF1h7GFRVkUw9RFqwYl6xgkNq6dakR0GwksQkyMqVaSOMyMMeUjx6IW4JQhTcapFhETuoDNFBCJuelDixVpThvx0Lf4HASZVSqwdsZU+inqY3Zf74RFXWDZICOX3LRmXVZFsOPWKMFPksAh8BvEEUnSt0IEGc6AmVMdj4oZYKVS3UKgUnVqMEI/BPL09HiXoB9J67bUTb7RlMnXiumW0d09WTPDSqOdCrcLE4oO2PXqG7U2+qWKQYs85pTPifHSc2T6ckhaoPZa0m78OphmWFD5MczMynP8Jra9HjVNMY2443vX4QNEgVEtfxHEXQh7mryAmm2tp6OYMviq78xbycwxdFLSxXj1iKdOFCfkIYhjaqdvuKBGEd9bdmBZr0XZ0X3g3JaGAmEMiFpcZU6OitCslaMtxU1PKaaywrMHN2kbcFZaw9ua80HTGeKtlkqEEDCBcsgVtZuqhTkl2zE5MAeIKepq3rhKCEcNVWh3QtjQU+hHB3IHQqK74E3k8hwQGRah8CFYDdKZjUsBzcAO0QMbfVw4jwXoxxo/7DtTwGWasVstCi89/ISPODVl0LooSvY5DtINXGzXCQuMmT3T03zKSDCtgT8g5JS6O0fBOaEt/3gw+MRFD0kKnkbGiI/5Rts/yhjc0TvjPHPcNfPVXgjtko5Le/gsYBarSIto8B7igkpVwxbu0HP4hWqwRZMUrftC2/jwCxEaF0tKs9kG1mVYnKMnpyA/IZBf6PTPea3qbIdfzI1DmMzLnBdw6AF88jyUuCVg7M5VITTRlitDR4meiFYfHxGMcrf7DLA0MHgsHjmMmFwE0SNuVeEDUOal77a9q9eSwoqOLkGfjpMKfyyZrSShAQGvZ1BTUF4z53W2q7C/p8DDECE+BEG99m+g3FfeiLD34OwCWFQNPgLgZhjgHSOqhN6J65LGpcy+lA0LiUqYtZd91UqsOEe+sOZeAO90bSpTCWAIHxkSD7x6j8UyY1ndeJTFcdL5YsACwNygacBIxcIDAd4uj5LPfEwHNqsOh+ApQRwtxFrncXwshY8p1oLXj3vvG/QUwEZPSbieka9Pe2M3r/DYjl2cjSGB/wOxQrzq4B4+VWOgyQsICO55Nguui78UPctRRjx0/hriw7+HaM1K9fmW41hGFXLkpGl4Q9khx7uNd0YTSowuJoJ4YBaR6GHVAxDAK1hCJ/f8GCTVT6x517rTqC+4Hb+gzdb0DVP3G0h6iAcZwI/e7HKMZA3BoDEjfZHYYrdSzfHahbo4kIc9xNB0+S2K6NuXaWGC6C44uKWxmafnR5v4WuHRnFmzYu7sC06jHkQD63qw5ISrXGzd+//y5NwAdqYzk32PaczDPcTF6naJjx6EignQOhYD6AVLBEV5I31NkZM0kylN3RBnFQzC+XyNnLJcbJ1wFtHAfUSfIbGIh1XsYbe/KwVki1BP1/TQWmgTBP+gO0F/Q/jk13qrIZqEsg866ouzvCVQVeWCXmvPmUp2icPFUwMW6gts2E+y5PgRGuHq/a8VWh85apSaoA49pq5x9I8o93//rRYU0bUM5A3qC4KLcrsAxCsG+sznB/ru1KYWiJKoxqi+7Iw6go0qdlvInqJaaipBgh0iMxFGUdi+p3aeW+cAf2TrvFGN1uPOw64E/QTwacXQm3AOMBYSIHJ5E24i2FrglPaqZLd9VNosp1pon6DIyPIXHcsHAXuJFXyPl00YGp9bwNNsBNCFCZuDtoJeYou0FgwM6ecWaTlmMsHYa9V2IEM6jptdwIjtl9Ekv/OhM4Em52FPlOv+EuysB/oe0hj/aIrrk8blaR/XEJq8ezdibxb/sQlXfKhEWHMJMPmvo8+lW+5B0nYdHh2wkY2SuuCNj1Wcn+dsR+l3l1tLn9rbtJ2UFhxrjr/S47f8qeMvCvlltwbfC3ENOwC5EnVCs1nNeepKekuXSHO5bQSwa/McY4OKftH1WevSMIb6Ky0kHvZ5uGpBJ54VQhyLusot3Q0mTueUOK3TZJqkmqAS0NwSgnJ9CVLIhLlLPJI857/iAVgGq5zobaEke/iSrwnEpeB+7SyOO67codU4Q0faPVxScg7nVnv4cazzEqs/GRgJ3yO8Y9QtrjyAf92fWU1zBYgLTQqqcs3pQ5NvX8azAxylpg0lUInLjmbdjupjvlfTGAJFMRwOWOsuRmINth4QtLPhgpdXrKwiE4PWXifAo6Mo2KCrynEYTOCdUWhu5oJRgJsBWz2eIad3zNJ5gu8hutrmHRPreOLQ0MC7GQGEHOU/CaK96FhYf5+XRBVjG0Q5a0LDUoUf4LuWJzeF8MNQ27myOqjawJ6Rn0g2qbFH1lQ9orYIGrirIjwosJY+QkOlK4r15BNxmQSRE7qMsmi4mSusq3LFRpxBweJVr4rd30DjQuSIbZMQMnJquKAl+UgfcBJCyJ9L6NwxUF/7AxA40wKMYv2FpYmXSmcCybbiyZzvQEKAj4xxBUHoxc9dGhvNyJZW6Ztw7JVVmXwhMNm+k7Cfm9wFW5XEdJirFBA9xQfWJr+ajadhQ8hQFYxR1R6qpYaYi8HA+yacrO3U76sGvz5xrocRvFW7O/RYUYUq5RU9VtyEtDg+Vee66d7+sGl/7ZwP/DHSTbzlrCMJdonSsr40x3PRxvT9CqbaKeJ4CpovBlROjDAlXYC/E81uzBDSEYqnjuL9kbTIrLmzYx1cF5BbNXhXFwTYCvh/vxgI9/djwq6OD+kzGjpY7US9uAdsjLQ/+ia2NzZCB03GfBf5KHpzy7EGwKY6UjE0FljAXTiD7S0dqFYUBPe0Xos+mBsjYt2WCm+1nWtrkmE5Z4CNmeVhZpW+kL8ENA4cEw0CW2NztIpVJqi0rXNarrWh1YwD0oEK3o18EEZkCQsGtxGXHwrWgXj8nddxcc2jFi4iw7o7F8262sd0pG882s0LrLGigveXvglr64XSNw2INKzTfhBxsguVWP9ZLauB1pCEIHrXeV7jTX4jEwotGeooU/n/15EUy/9p0vHe/y9PSraUenEVOyZa4cLKuxUMJHSVuebjFwWCwKW6XuIugIN6tb3PESRhdc4Aauma/Z5KvVoZ9OCuQ7lhnNw8CkM2ElWT2TFK3gcZu5q0jgLl4I9TgSGn+LOfZ4mqaiNX2tsp7Bj1Fquk157kehtKgRbYdcYAcheynKRCNgbtxkHJwlcEP1aSTepb4co1PLu29hRYKVpob3EsRKVKa0r6pXmvMQVRRZK8umoMBxqaTltZPUFW9YgkfT8K5BiNkQ4FfoVHBFFpDPURr2MdVoWicf3NemYzIVsZssd6roHrP0tLh+knWI2x29gxBHKG7LIGPZDl3v38u9+Nc/TKISgHk1qd/Amnp+6LOhpkOTpUm27eXK6kbaP9ey6IXIbDth7nbMGHfxLY1qhIn/BmoQJWYUk0BGxaQZ2u2bW7QUz8TUNqXORHZ2tNeO8XQm2nqqyOb2gUEGkEcsMhtKr7ceLGONnbHB37YzXzqmUjclZkBvq9GxwPB7XYUVvlnTvQmmeKUtl6xOgOs7SDHr6jDnR3eVLOHxXO7NuG01YtnCZI8puf7ItW3kc+ABHk3uI4YaVTIPZ53AyiI/Ax0JLYpGk8C5ug7MqxN7Ixysw/E1WCRtO7MhpA/RzSkxy/ORKWUm9vrDfLLg1IzDdZLFuGUCvov18ZoO8q3QqL8GPz2WYr64BiVSYcKwrrdEI+lJ6aaHDQjENikFPS4N4xtjRLTHAzuGhE4pEIpKapVdYArWVJOOFyLnodol7KLimcSOzGCCCGFOOs4y8EnbMYLz+6zSodqIgEl+wF3LNNrdriKnmrX5FePZEX5XjHAWt06raPMrsJcjAEba606F6f2T2uvvYAXgpis16hlkUMKpKt+Kfq7KbJgYqye2s1CUzeStXdwizmFOmsKpc2ePAE86ztrJ4tBm52O7hBJLM5ADbOZTpiplmJLbVm9Au2I6+qgB797gvonK+0qqZzd+Qud/8wb8gyc+FIT7YHxyjTJYCcT1aB8fy4R1HhIw3G8lZ85KwPQNpZm1ng9nFBg8xjqxNyNImFLICvjOoc3yGZgUGJVx8rXyqyqugUkdjreZnm/o2Mj99Pz+yg9GuyDyeh0SvTz/2ZxxCWjhtpkbjhc5vKPKQm6CMzr1x9F/yVOALh6Ab6KUz4E4dKQ2/CVzzzppjsR0Kt/sOYVUMDutjHTA0IKyXdqkgYD4dnBAeKBXYDLSFeAh5r2tZCKx0J2xa805CsG8D2S4U041OWY2Y0h9+MjI+AXlOtWYz8bCGH0H63h3vDE/sjz1H6b5CNRSWJsHhJziBq7r+yGGakvPv6asnm4t2sLlWuroUoi5PoXnH8Wtt5u7uV/Mp+dXCzdAJJ5DktKV0InDirjbcj49864ILBZMcF/8XqWETEb6JwYSnNreclRA2eWUmD2KMlVCYYFBKFwzqoCOdGbPoQufw2i10tnimp1wOw+Y22N9yzi0UvVsamdD6UTlluDtR6D5iIeGFNY9jfAHaWiNAdnfRtEKLasDrUUEX0/AWiAwy2Hiw2BGh60Fu3uj9sj7G9UomZ3oLkLudF7+8INTy8xRqcwg4tOGdpaTXQH+x6qVnmoH9khIxv0vnJ6djBjUuZLvMNFNldwmmD5oDntUzia65zMIaS8adAz6G9IUOpURU3MN1HxtAjDZ3QV4qujOgrDEGFK1ftJRrIdNnupDZcf7+QnYlhLOKpMHEjrgZOLeE5DhNpVtFi0N1Kk2eZOuQG9spfPGodAXis+jPSilsT9R+VIns/n+JFmdzCbByQf4P/yP4KTA36vD4hBQiI5rgs9VtrNI8vlIF79kVob5d2be98nhl2zfkemxEug6MzyIu0cgNLsNmFznX4xLeZOdwVJ+cN8DuKj9YLHVhqzssY6Vu9AGm8ZdBGX3anv/+hYKtgMonxTJ1SZ0wpVp+yk1vk1vij9pWUv0BwxuAcf9OwZyoKL6XeO6PwG9QSlmEmZeWFDpnLyuShzqJ1VPKUQNzFaJt0+4E70/dAP0JYfmqfrQL7U7UGlpJlOqZAQT8IEoZwpPmOBH9wMnPfFLwS8jMpTwsQ/AlpzDE9g7pFGMU41ZkYsu3gkL0iPMPjIS7eFRt0bfYCQYhEOTGbKTaY6rFOuDZZmhxQrcAzYnfNwkdxsWL2G3C+o6/GBUDXUzTxZzIMdCaa2wGPlcDAympHPuglr631olxn/0Z60rSdW6cL5wvsfYNh3yqJTQc+xTtRzOBpu62VHCP8ldIBBYve2JjRzWRNofLaCoRqTcF+sqF3Bg9KzppcRjsE3OP6w+jUc38PGC9lodzgMfeHFBO3gxV1Bjvz0Oo9ln8UxKtsWSKOX0yx+UA6Nuc38W/w48B4rz3HISmPfxtBAMFHd1yqq+dl59/w4Tz2hfo+JDJOZgIJ/KieIt0n6Lgb9BKge7rOAS4qyohDw9vq4vXoUtLn2xZiIhreNvHFxdBCzyfCygj40hc0+H/Y4wRi8IRRaC2CcwUv/jsYXhIaDDkVEruo3HMLjXOXWKChQ3kiRvOSvfpEulY5eIsEoza0bv+SA76eVjhRXsGJDK9B4NKNlJBsBG+GsuEcBDYkbYKJPDrNAlHVrR67QVT6NrYsQ7sLfHdWW7MOhulusqndIRuLCg2uSHLocEbT6K3nPXFdR70Nt715+7xWO96ujssjt6Ux7oKWKyVZbEwwwuNbv84wckKk2YE/lHXRShT6P3jvnRFLAxvoRV00DzJzosFonhSbQeJDz5NzIinO0erDQ1JxmF0O1HjjVWvrV02gOLF9PRuxcwiKsOXrZQ/fYEECaPcTKHHnMvxaPMH0Cy21tx3TsOzLF9tUrVbQDq3gjXD+/S/NZzT9XdLX73wAFC17kI2hyhaJABg+d1q2Wag3SNq3uAR1dcuQ/glMuHNMmkcEd2kNBwXVvrm/foxXzbUhO7Bs/c3ETQWXju4pq2hEoBnYavgAH+h169dcCggOOAjapIXT+R3GXARvZ2ATfnLTG+dAvDFFYh9u/hf/ouEZVyIPbl3FymsDR526XGmZIogFnwrPVjr6YQg7ZoMUI1FVqHGjsC96jB8QzurEC9r+QwcOUuolvj7HtlVGzfiutbMV66QUPMyyHGZQcFhoGoDfBFPleUsPQZgFHppqza1YZERgYDd2srTZpk+cTXFfAacAO3s+jgvbekoYSTCQLXFnPwOpBsUNaTOFDSFWODpEcYAtIgypZ0/GVBq76kMzgtudRQLqxh0V4BPbKiac/9dyZJm37QjT9cR1zt6d9ZQsfWA+X0GTRgRfldPj/O/KaRvswm4sNaFrd9SFaWbNGMYZMWG9GS0ew1G+e+3joBwIcukNa0VidnTX2sHJye7tfufndY7rcHd4YN5ruFFs/28mkljPcJHGeYc5T1/IN1gxAjac+INcEFgNUo/zuz/NGZZix+zzRzi2P31GjO5Ctj+izRHrTnYqO4mkxfDNTJbuQ0CGXccKbLny6Hqo2/o3L79NRFpbHUITh9lc434moysrPJt+s5t7J+kDJzpiTrribmvibah8AA3Dp5xDut8jy1jvNgB/tjFyUd/BcdDEZ6f9XglQcY3VKXZb1+ZZ8WwnNv3XPO6jCcui3V/tA7YhanOW4a470HHQDthVfBdAwjvdOrG/TuVmTZTg4QZZvqiRR6Qtucpd99SRRq1tWx29mIG5bLdZLK5bJ7a2H31jZ174JQi3y0H0tkPHvRFT9Yl2xp7PtUsawu6yZR/dk3YrPeCHVR3vh1lM/eXzVIrXohNJiRicRzp2WCu4XdKyOfS53qJ9eYAfDnNI+3Cn+Y7hBfXUuD4buSZY8sr6DfVSpn/MOChvIYKKVZAe3uwLU3zagF0QlxmZXU3zLAHZ015pn+9PPPP9yoK+fOpoeLvT1rB2fPj3OMWC8ObrBOm2rTT3vGP1AHIFi1osAQGkL88lKICSt39Wk+m3W29vUYWuOLII1EAC1m0CY45uN1EFzuGc4gLU//DdLz1A1Dn56Qh38jMQj9pxw4oRLXvV6C+oXShhcKbVpIF8rms8uC3sEk+49xZmdPcDKLbkphfmg9ktYyDusL5z9LjDKiJwosyZHZMl81sT6B/fL79zdvHb46V2mtivPqNiAxM974fw5RcwMf+aPqLiO3jRm4CzNyc2naYozsQ5C2KdOhXteYbmWsuexwpMfxDgdqncj+cd7j6fk4UJIb5vKmfkW9Uh1HQZ9dXlYHtau6V7Q86RhjJ4tZOPnyoPfY2lo9E03XU460qcbvmAPCIQnzgV6xXAVJzAfbsMPvGJmw4LVxCvyot1b0pooFp+tmPC9vmBdaH18Z1hj3Tm35SDJWZS3uEtrZ7Fk/5p5nG9j/A1BLAwQUAAAACABDmGFd89joXUgKAAAdJAAAFwAAAHRlc3RfY3Jvc3N3b3JkX2JlbmNoLnB5zVrrc9u4Ef/umf4PHHc6IB2UkZxzEivHzjiJ8zqf49q560On4VEkJCGmCAaA7Pg8/t+7C5ASRVKPOElbfdADwi6w79+uxKe5kNr5qES2M5Ji6uSRnqR86HD7xRl83CneazbNRzxl5edZxrVmSlvC8pM/FfFlSQ7c4jl9LIVS10Im4ZBl8cSJlDPc2YnTSCnnVXTJXqScZbq348AjYSMnDDnwDENXsXTk9fDZl+zTDA5RQX8w3xeLWabNJprDTXLt9STTM5k5+535pjHLmIw0q+6jirGEpnzKNUXhcMNMMqriCZtGwanImGevg4+l8/0oz1mWuMWB80185NglX+lIanXN9cQlZ1LkQjFS4YaPqyidseCWTMUVU6TXvyUqFZr0yKRLKEFNwfujs7OTY/gYi2zEE1AcIz3/8I46i937i91nx0fnF/XdT+8Gd/ODWapYrzwZBObZ2J7Ngb5DySckoCSHlyd31C537fITu7zErtB0wmPtavZZB+hKfjKb5so1p3iUZ/lMh1pcskwF+x0qZrqy0O2AGeC6iQr8LtVylsVgpiR4FcFFvdI/PqDS3bmP4ccXkSrNgwbG9VCyK664yMIoS0JwPs1kpgr/mV8ZVaWCoZ+nUcxclxDq7cEtQHara2/Z5HA8k/r40yxK3aE/FBE4sBi5hovX7x0MmnTLJ5hP9ICS5+fHRy/XsTc7+x3guG4XIf5HwbOWy/CBMxLS4Q7PnKEfszRV7oHnrT35rUI/d5cvC7o4eX/6+viceNsTLSQEsy2ZRcVCsroZwNR6EpCj5y9eHr96/ebtu59Ofj59f/b384sPv/z6j3/+i7Qd/EHO8FjL0HCg5tnrEyXSK5aQwVrzWUISkgeGrN/tDeYMTGqK4ngmo/iGDKj/dDtWe/sHcx4pQ5+rcunUlTGcJWOGrgq6uzGOqlgk40nDTSF1zHMueDwm0UjevOSSxVrIGzfhMsD87IYhbglDD9KT0YLr+XkkIZl6mGT1NF/OO/FoHNzu7Q19UPzRLycfKJlGn0P2OY8yDB5IB4/ulgjQqaZMT0RiPevn4w9v3r+8WOZqOJsUHiyyues9k0Joe0+4iPfQ8mlQKqbwbAiaC/vOtbwoXJYiC0xB9Id9r0EJMs9STUMgtcL3SQx5hwwwqO1ptOCOzJoMKrZ9LRkkH+lansAoghCyRlxDdQLcrVeUdFroKC0yHJDDsX2j43KlTQp0h83iPxsysAYLUlZumdekVarZX68be/JG1VQFpAXjLQja7kmtDF6bU6K8VVX19g+WfbHNxdY6z8L1iA090upKJtwq1z+POLCFOH9uiC60yBGDGPaw5dIlE8ivApiV0ILQbs1PGtqoaYL2B/XsMI1S0M2UJSEW0npOsMXweZS4C/lroKIJdvYiurd3eR3JsZojo0W9Jpmw+I9sKNQHa+v0khq/S9ZqGhn1AM40T2NA8mWmfZsBROHJOVM5pD329fYt6UEOxcJRxFPAk4o+2pLMpJutd1u70CcHdR9KIg3YSIc5zzKAd6bGZAJe1DXUJmPlumOBdQxmKWA1PuZIQkbZmLmPOzVHM+AjSLnSLjnCIgi5CZcAvwTxRLqPDx7whw/3H5fL3cXyX2B1iVdxfImq+9U0ZagrGRPFHDHoK9yZTGt3AvhNHsZiCohePSR4edhT+nwFmQJ2nkQAlyOy90PnzvMBKIsEHLDODKjhu6SA8SdvXxyfXgDQKzkOScpjsAEjNWnq5xXytRz0LeOFmnbLF8OPQOsOKTFqIlRBJxCy0QhWA7NU01o+++OPlBnACsUAWJYeZNLmemfE/F7Qe/RgU3zg7tu8j02Fhao52qigv9uCQbGVtl91JfGp0G8zFyHiTHNMdcY2qYgS5bqVAmHSMkKgg45vciIqO0pCzJSu54Frb5VZfsXG51hKIb1e603p4WE9aCGhQpoJsbJOWdG/gBMkCBShe252Md8t0Zbu0DdFgifBiHy65XeEYkzCciU2S4UG5N8mA9STBiTMQVuVXwM9uxvL/dL3f3ZOBQRclMKhIyax43UEom/lNy1l4sMFGMBT6HsVWIZnut6VG9eZTaegTIgIMMvc6Sqg4vaONmLDXBdnEduBsy/i3UQSeBBtO2gloemdoMK4qk8yMgig9Tb2UmivQuItqK2pHYQNGLuLMN76IotcYKpHNQCLRAlxN07F0CV7RRhCE7tfyw3TKOMjOChYEcvl922BvP5mJWWfWOuERtmA4zfW/wVlmUlCnmAH0F+V9AbrFV4VKIf0ULL1Y3UFMrHPHOci3j2yUovztcHvw8PDu1aP/NJDzKSkxcuXs+CEzSSIxONwxMwwrpH3sExdaAS3dsrilhMy74Ed5RzShOV6EnTpp8B/SvPAf7K4+QioF5xpHyuJ5eYNKkiy2+lUTN2CwLQ78ii+zGn65Jrx8URjd9dKOg+gzo/B1Y9B17jCFbrCyDeTMrTjakqsI1xIrm/MvRu3/Vtlw1ymto11lUMKZFLOcgDzIQJQeMoguMMJooZYqAZWLJoQiR5y/z4E3cU5h7jiU+svkJejGFgmToGbyf+oq7CC3b+xOGdj9tmtikabon15q1G/Z7UkftO7os0drpxZdpmJ62y7u9YGfhOALKEJUANlNJg7xVYEasxlAkxDBESAkyUHr8Md8QT8giWNaEdGm0bkb96fvFw78x4Lkdxrzr7Exfr9OVSoKwbAYfwlvn+fHxyWMkH9hwe3ytHSe02MYTr8fqURQU14D8jvv/9eYGC7bNTs0frGQR+L9NL53l+7g8YxjaECPn3d3P/7R36BLCv29J6p5ghpc2Q1y4MNlLz41acaKo+gw6FGt+tZAA87DFBlm69qQwWPuo/oYYfub4OzbiW036bgyBakBo1Xbeaw7ZBx1dTyHkqzrL5ecwWfQmuH23SkK+eT9R+XtLzBBi0H11PlHB+TV9Gilbb5RlM7qJOVgrk56Mjt3f/7DK+9/bNT9nrnpxqTPoSNq/xp7a8AOI8Pk5EiVDXn3Ctm3H2itMjBtBG2DwPkASoMhbQgqRglbwrj+g8J7T8QQCx3KbTP/xXAc48BaqFcqOIx26jgObv7qZnbmWw4FQlLTbThaHaTout6Xj1DxTkQQ6BDrDx9KGiNASoESYJwBWJFIXbJMPo/mrCvQ5sS0dSD3vwpIp3iMB9iLIPXmMnyfxHvQNgLU7TPMKnLpeFmFMfQwCgT1PVJndkd1MkB8JeKCi0WcBeZs1YfsAjEE6wCyL+ZVvgIvy67e8PeN3kOUN81s0AtiqFlgHalnIOa/NHgVFy24BElFVo3ntQ9zDAq9kIHEiLIWewxCI7c7hoItwsQbhch3G5vd9LdpbtoDXhvIBx8XEC43V7HYLjVvyWXyjaVpdIn4hzLeiLqwXxdwCb79rdsBjx+y14IiSHpvLt4f2q/800Fitm8MbX41KMNBVWFa1I9P3pDvBpHA1gnj8Gm9S5mLpWxxVysQgb8E4Z9s7PD8Z81WTSFnBEEJAQH5lkYkt7iDzywAKr/085/AFBLAwQUAAAACABDmGFdHvGW3TsNAACTHAAACQAAAFJFQURNRS5tZJ1ZXW8bNxZ9N+D/QKAPm7T6dOosYCMPqZO2bpvUSZwu0MVComYoidUMOSU5kpVfv+feyxmN03ZRLBBE9mhI3o9zzz2X/kK9sc6qIvgYDz6U8UrNZ6ppP32qTBypdPAqGh2KrWp8ZQtr4vnZ+dkXX6gPta4qE1RonTrYtFW1D6Z798n+2VN672NTeV2q5U2//Xy2+N60wcZki8U+Lu79/eJHvdlUZmKbo1stJ+p+a1Rp1rqtkrJxYM75mXYlm1SbtPVlVE8uZir6ai92xKcT9VrjePlabUyKSidYFpN6PlPmodEuWu/i6PxsjgcFXMAb2HR+MRvNZjOVfNKVsq5p09S3CR94tDMuqgZniB0T9co0cDiYWlt8g6XnZ6ugXbG1bqOe8Ybr4F2yWPN8xh5FozTiAxMqH3Ty4YhYpYQFiLLzOKZ1plS+SbbWk/MzCkKlwwY7kJVTtkKt2lKcqip/kIifnLpWmrNRaKcQ3qqChYgGRc2nLTaqbG2T8gFrtrpFTCy26g1dmTXt9yhOE0ri8u7jr7/+9Hrx0+2b2/sX89kSllemwFrsqtY2YKcE44xLAfhQfs1fRF0b2v0TvrqcjSV252eN99VEfePTts9iMIWxe4NVSJYtsY+Fzyq2K4Room622m0M7ykeYNNazZEY7Ht+Zh1/Veqk8boqDBzPvsSkA4X4OlsMl5QzhJbSNMbhaDyAqwcTIsUlFlgVc/TXLTYia/tER71HirCzaoLfG4eMAwxL2duUi99bExOH7bfo3ZL8Isifn3E48lsCLiT9sLUV0kd4pazV2tk11ner2CnzoIukbl8JSAvfusQpeY8FcLZtRnidveTXdyY4Uym9Tr2LxhVHHIl3qkqTcSN6EyChrLuWwhp8u9mKkU0w68putgjj1hS7ifqAiC7ff3y7+Ob125vv37x8/+OL+9CaJQJXDF9vdIx95Jw5UIR+g7/4AolGFS+nO67yKVhgh5RMiyElRBxX68X+2fK6L304hg8qemQSSV7mB9P+5YXDUoD1OTYwpvz6Ygr2eP1A3II6jL4NRY8LiWDtS1MBg5RN2rbwjTWlAMpXpQm93Wv+NSJLCJbea1vpFZX+PWel8Yj4GvlzADlBsKja0kjO+kRzuvjYvqAEwMRSoaKaQ2Galfe7mDE2OKmj2RTaIrWBuEH4KJhG2wCKvRCKjVIawWyMM6AWvNlty4f/JUAyKihWPVZsn8T9hVrpAqRTqhapVcuqHgP6tU5j4/BDYcKLF7PJfD55tqRSBKBiCuTDDx9+fqtKcDxvXPgSP1ydn9UoGRUrT5zjmb+28/H2crqfj/eXEiaqBYIEo5NLjgugOiLWWPzabSobt6oCcSI3MPTG1w1+g8t8KJiPqHSriUxgC/tW67Ar/cFRgWu11qsAdpH8OFBB5KrAXj95Ip29rmzJj0Cy0UsdROXa2mBdZohcjXL2SLXOovSVwdpWlqJiKYyoaznKU7R5a0pHg1BhEzazbSh41NOCEfqkZpo5jSn0RP/oMsL/1x2Dm7JfBlgQKJf5nAVDfdGdtuTOg/6NYPYMkvpa7at+SJ7Snv60/CfqFmnkI6SJSmsu0GilsIQqI6BKiKQEUonRgbXRkeEMgxDS2jCl/SzY7kGEbTLKlICOQ45626St9EskIlChZvpm1NrYVcdWUx+CV84MUkodsiIkieJQ3919ZO7Fuq5h/t5aGDcZiBcyet71r1NpUeWXSq8I1chg1jCIizL1ypTcoxHdwsSoqdkLG5EXnV0JhRnBBY7qnW2hb29doiJNI7aMSZ7TEjNxx79i5IwZVgB4f6LeevUzSL/WBD9EpVQv727JVweyhI+QMcK0nMcr9e5g3JT+u5hcjv/5zfiWcgHyGamvxyu03bfffk390js2VqiwDwcFnNKyMqQEHkzRMu5dF+rVkdWGbtHymRn0xuRm9jrX+OWsq6RNsGXGUUemVBVe2K/ShSHYnDCFHTfWoXqhJtWyhqCdXc4upQujPCO3g14BUK+AaLt4Li60Ddw0uqaCBjmPuGOQPwBaf3zXgnMaUUnQhOYEddoKkcU/rQoUJgyDICzxEVF2SSAL6/4BIdmuiMWIgLoWyFDo9Q7yNqCSjDvsDBoW9QGpCI8m6iVDH6R+lGBxEy6ZT4g8Cgn0GiTEO3mHIAv1Z5kO7mgDCUfu/aTvj50Yi/I6fUHaDv0NnUG9yiIrtykQbDqq0hvJPTUa9g1lEQ0JBM6Q0ERDRIXKJstzb3uT9R8HdquDcHIKvuIhY6yWkOOLch2XVxS7cUzHikYDiO+xyE7xY0TL1iQcCzNGOA3vBHlVBgO9I+evwTs2W9wEjiAfTE0OhhUkSjoxRIGBqVTfFVUjMrEn9085rqhZjGVnXeomZW1FYdAu6zZgNfiyzdFf/yH/+LZuIGiG2Q4++QIKmd3f9pPSCrFllxGK3BVcSayWm9FpC1KWNI7Rij5KHfRXx6zZ92a8hokAU3+G1CK1YKQxGbWFAooi7HPrkN4MkNmNowJERN6bveXy0rlxV8hI5HxgCfcPRjm/Mlz4aABoWcMQC5NrgfeEtaGt0F3Llk6HRWOxC7ILORkhcQm2nhwfnWKishTipxlZw/wjzSaSfzKqEXWMO3Wl3Y6L61uS/0JEp4Fqla0z5TXZUQlmhLCYWSn9XWtyoH5QMSaQNdsvdMOnyYJaP0AOfiJxVaCNBFCi+lK9Wypde2ycXSMs98nOye1eH6k7qZ48jHNtINAKE90KpLajzqi+A6hpcA4g3zz/FFvvYxasB+u4GiS8xC+g0w2JX+QSHAA2a5qubLuewUJWpuI8bZMdQlZXfzZOf/VonB6pfv4efTaaZyewg/qfQzW3DdkOMQwit0kARBP2ool5hMs6TRQdZCdNbZ3GYcVJVoBMxSVEW0QWsnNUpH454K1rhVrJ+IzeE8gykRAef8NiBjtNF1sa4NEbbrP2E5EKJ6EmI/dD5Khi2ZSlumg5GU3gTlYi4Mr+0oQn8u/VCzV5dqneqa/wOQMKxmoyu1Sv6HN+qW7oOT5/WQqRvrsSFhzLfKAz2NBK274oQFCdpGXBDUCVzPsi2kdKphxyjsc2ZfEyTU8gla7GI/PW3ePThpWHQ0hwx/WxK50D6tCcWuGJZHMtIXAr2QfcuMo9h495dYV3oA0rpB8+CcMxeMD5wKjtzVs+mY/vnmK27KgldsNZzOvE8Jsr6mG2FrNbGrvGNK/S0Q/HK6BAOMyUY4oJ8JhPxRoqJqj4HNox5zTju7R7NCe2RApieJkD2qps00jf/vdsNP8Pi8g+ANw0jWhr6v6m0BEi4aWcE3NWTG4mw8mEDRPAknO/ICtov3VbPwpbT8owEwzQd+3u0qhvhX3xMU/8IcTx95CevPsSYZ6oD0guDMD5/SLEAIxCEIBnKFfPwP6XobEjA16HQJsRtVFi6I6Fk4WAHeS9a1g+UE3yFFCVQcUyHdG4QDX3rdGUP9kbqmU36NP5yg3oqrStCZTo6CA714myMS8QRfSZ0KXgttR2cVY3FNDIgaWlXa+hPRBCSkvulkXRAiXHTNxaYZNEM06DV9DHQAzX3GTzHae4jXyWVm+cp4pHIxBtw5KsI4Q8MeUJv6VriPOzx9dqLWk+4bhsaKY9jOJcfj6PgA+JJaANWbmzXiEd1esfUj8fNBU8S2mT5TlesfvuAkXOGFH7oNs7UtncExiY6MN+cLeXpU/u/HnSFcXMbRPhWw18magbuaoNJt+PmXKa6XU4ZZ7aEYcpxwjND8//5g3U9O9cM6n+xsc6ru5WV/nCSkie7oaAsu7OalGhoU+KuMc4soSR/VVh9wwJrDEmfv5rnmCCPgyHbWn5IuFJunJyZAV5H2xxooXPQChXJ4MH1CI7GPNNOuDE5d316cxhABOR3GjQqdmMP15rPKEA5OhNk1yBEKDWGBmoIJ92xlKno/zzoNHBv+hvdEAqjyFw63J1Md3CkJwCiQzP6mRRP46Q/IYYcCmzLozLspJdkWsQE9rBXUs3PFNpSFUBeb1FXWHx1LcT0uZUtAWN+Qhllx+GKbrrMU+uubr6OZUvShAHnibzhDLtQ9xdZQuXY7LZkJRwG77l4QsKusgiCdvdCZaWBKcHYtQrf3D9vYWIzV9v76Z8j9V4cE6eh9PpIlOuC+49lxazLw+JupvdKYooYJ0SUckRRiu6T8O2FCfNd0hdcOW6k1BJf174i9JbDu6Y2maiPg5ZoMEsRuKE+0UHs+/uPnbx4RF4MGNRlmSC7TmaRnKI6sY2C4TJfDKT9JDQl15mv0AcRPQ84gfDF5Oa2hp8XeKUCaa73VJwyfe/6DorAG9LV4k8/xU0EJJmpHwLkYe2YdSKY6LtpCT4eonvw+iSpHU7R9EjCFyzLyIVgDKUROTmJBe2XY6FT1GV1I7TloYRunmmKxO0bhnyhybQTRIFmgEuEItXaptSE6+m0w3g2K4mOG/aYGlBbXrsqgalajA9r8d0z4K2Oq6qmkP//yycriq/mtZQKiaACoKlMffEt1Mhr8XpyRhTvvwB8Pzsv1BLAQIUABQAAAAIAC+YYV3pNIPuJB8AAJxeAAASAAAAAAAAAAAAAAC2gQAAAABjcm9zc3dvcmRfYmVuY2gucHlQSwECFAAUAAAACABDmGFd89joXUgKAAAdJAAAFwAAAAAAAAAAAAAAtoFUHwAAdGVzdF9jcm9zc3dvcmRfYmVuY2gucHlQSwECFAAUAAAACABDmGFdHvGW3TsNAACTHAAACQAAAAAAAAAAAAAAtoHRKQAAUkVBRE1FLm1kUEsFBgAAAAADAAMAvAAAADM3AAAAAA==")
assert hashlib.sha256(payload).hexdigest()=="1393fabf955a9658b7363af3b4c7dd907ac5014c6b7ee980cfff3876b451d821"
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target=(PROJECT/name).resolve()
        if PROJECT.resolve() not in target.parents:raise ValueError('Unsafe archive path')
        content=archive.read(name)
        if target.exists() and target.read_bytes()!=content:
            raise RuntimeError('Existing source differs. Use a fresh PROJECT path.')
        if not target.exists():target.write_bytes(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
print('Project:',PROJECT)


## 1. Check code and select the fixed puzzle subset
The tests below make no network/model calls. Dataset preparation downloads the original source file and license; it does not run inference. Selection seed is separate from the LLM trial seed. Keep selected puzzles and heuristic settings fixed throughout evaluation.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','test_crossword_bench','-q'],check=True)
sys.modules.pop('crossword_bench',None)
import crossword_bench as bench
SELECTION_SEED=2026
PUZZLE_LIMIT=10
if type(PUZZLE_LIMIT) is not int or not 1<=PUZZLE_LIMIT<=50:
    raise ValueError('PUZZLE_LIMIT must be an integer from 1 to 50')
pool=bench.prepare_dataset(PROJECT/'data',selection_seed=SELECTION_SEED)
assert len(pool)==50
puzzles=pool[:PUZZLE_LIMIT]
bench.write_json(PROJECT/'data/selected_questions.json',
    [{k:v for k,v in puzzle.items() if k!='solution'} for puzzle in puzzles])
print('Selected:',len(puzzles),'puzzles')
print('First five IDs:',[p['id'] for p in puzzles[:5]])
print('Source metadata:',PROJECT/'data/source_manifest.json')
# Do not inspect solutions to tune the policy on these evaluation puzzles.


## 2. Load Qwen on one GPU
Uses `Qwen/Qwen2.5-7B-Instruct` in bitsandbytes 4-bit NF4. A model revision is pinned on first load. JSON Schema constrains both generation and evaluation. Move words must contain exactly five English letters. Scores and rating IDs are also validated locally; at most two retries are allowed and all calls/tokens count. Formatting constraints do not guarantee correct clue answers.

The same backend is used for both policies. Results from NF4 should not be pooled with older Ollama/GGUF experiments as if the backend were identical. Model loading can take several minutes.


In [ ]:
import torch, platform, importlib.metadata
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
assert torch.cuda.is_available(),'Enable GPU in Kaggle notebook settings.'
MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
pin_path=PROJECT/'model_pin.json'
if pin_path.exists():
    pin=json.loads(pin_path.read_text())
    assert pin['model']==MODEL_ID
else:
    pin=dict(model=MODEL_ID,revision=HfApi().model_info(MODEL_ID).sha)
    bench.write_json(pin_path,pin)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=pin['revision'],trust_remote_code=False)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=pin['revision'],
    quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,
    attn_implementation='sdpa',trust_remote_code=False).eval()
client=bench.HFClient(model,tokenizer,torch)
runtime=dict(**pin,backend='transformers-nf4-json-schema-v2',gpu=torch.cuda.get_device_name(0),
    cuda=torch.version.cuda,python=platform.python_version(),
    packages={name:importlib.metadata.version(name) for name in ('torch','transformers','accelerate','bitsandbytes','huggingface_hub','lm-format-enforcer')})
bench.write_json(PROJECT/'runtime.json',runtime)
(PROJECT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Loaded:',MODEL_ID, runtime['gpu'],pin['revision'])


## 3. Frozen comparison settings
`H = 0.35 Q + 0.30 P - 0.05 D - 0.15 C + 0.15 V`

- **Q**: judged plausibility of complete clue answers, including implied crossing words.
- **P**: judged feasibility of the full puzzle; uncalibrated, not measured success probability.
- **D**: action depth / maximum depth, multiplied by `(1-P)`.
- **C**: future-work proxy based on unassigned-slot fraction and observed mean call tokens, normalized by budget.
- **V**: minimum word-assignment distance to another retained state, gated by `sqrt(Q*P)`.

Both policies propose at most 3 moves and batch-evaluate their children. Revising a word removes conflicting crossing assignments. DFS orders siblings by proposed confidence; best-first ranks the global frontier by H. Both prune at the same P threshold and choose their final output by the same `coverage * Q` ranking. A filled grid is never assumed correct.

Default per-method limits: **120,000 tokens, 160 calls, 60 expansions**. Depth stays at 20, branching at 3, and frontier at 60. The larger call/token ceilings give the expansion budget room to operate; any ceiling can still stop a run first. All retries, generation and judging costs count. Ten puzzles across two methods means 20 solver runs. This is a diagnostic subset, not a final full-dataset evaluation.


In [ ]:
import copy
config=copy.deepcopy(bench.DEFAULT)
bench.write_json(PROJECT/'config.json',config)
print(json.dumps(config,indent=2))


## Preflight: validate generation and evaluation before the full run
This small synthetic check makes real model calls and saves their costs separately under `preflight/`. It checks response format and legal placements, without scoring against any evaluation answers. It must pass before the benchmark run.


In [ ]:
import hashlib
preflight_config={**config,'max_expansions':1}
preflight_clues=['Fruit often baked in pies']*10
preflight_key=bench.digest(dict(config=preflight_config,runtime=runtime,
    code=hashlib.sha256(Path(bench.__file__).read_bytes()).hexdigest()))[:12]
preflight_dir=PROJECT/'preflight'/preflight_key
preflight_session=bench.Session(client,preflight_config,preflight_dir/'calls','format-preflight',42)
preflight_result,preflight_trace=bench.solve(preflight_clues,'heuristic_bestfirst',preflight_session,preflight_config)
bench.write_json(preflight_dir/'result.json',preflight_result)
bench.write_json(preflight_dir/'trace.json',preflight_trace)
PREFLIGHT_PASSED=(preflight_result['calls']>=2 and preflight_result['prediction']!='_'*25
    and preflight_result['stop_reason']!='invalid_model_response')
print(json.dumps(preflight_result,indent=2))
assert PREFLIGHT_PASSED,'Preflight failed: inspect the saved raw calls before running the benchmark.'
print('Preflight passed. Structured generation and evaluation produced a legal placement.')


## 4. Run the selected puzzles (default: 10)
Set `RUN_BENCHMARK=True` and execute this cell. Start with trial seed 42. Additional seeds repeat the same selected puzzles; they do not select more puzzles. Both methods run for each puzzle, with alternating order.

Completed methods and successful calls can be resumed with identical settings. A failed/interrupted inference whose token cost is unknown blocks that trial; start a fresh output trial rather than hide that cost. After a hard session reset, a stale `run.lock` must only be removed when no run is active. Save the export ZIP before ending your session.


In [ ]:
RUN_BENCHMARK=False
TRIAL_SEEDS=[42]  # Optional later repetition: [42,43,44]
RUN_LABEL=f'schema_v3_n{len(puzzles)}_exp{config["max_expansions"]}'
if RUN_BENCHMARK:
    assert globals().get('PREFLIGHT_PASSED',False),'Run the preflight cell successfully first.'
    for seed in TRIAL_SEEDS:
        output=PROJECT/'results'/f'{RUN_LABEL}_seed{seed}'
        summary=bench.run(puzzles,client,config,runtime,output,seed=seed)
        print(json.dumps(summary,indent=2))
else:
    print('Ready. Set RUN_BENCHMARK=True to start the benchmark.')


## 5. Compare scores and costs
Word accuracy is the primary score. Also report full-grid success, letter accuracy, token cost, calls and runtime. Summaries use only paired completed puzzles. The paired bootstrap resamples puzzles (not individual letters); it does not correct for related clues across puzzles or replace a larger evaluation. A better score alone does not establish SOTA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
for directory in sorted((PROJECT/'results').glob('*')):
    if not (directory/'records').exists():continue
    summary=bench.report(directory)
    print(directory.name)
    table=pd.DataFrame(summary);display(table)
    paired_path=directory/'per_question.csv'
    if not paired_path.exists():continue
    paired=pd.read_csv(paired_path)
    delta=(paired['heuristic_bestfirst_word_accuracy']-paired['tot_dfs_word_accuracy']).to_numpy()
    rng=np.random.default_rng(123)
    means=rng.choice(delta,size=(5000,len(delta)),replace=True).mean(axis=1)
    comparison=dict(paired_puzzles=len(delta),mean_word_accuracy_difference=float(delta.mean()),
        bootstrap_95_percent_interval=np.quantile(means,[.025,.975]).tolist())
    bench.write_json(directory/'paired_comparison.json',comparison)
    print('Heuristic minus DFS:',comparison)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].bar(table.method,table.mean_word_accuracy);axes[0].set_ylim(0,1)
    axes[0].set_title('Word accuracy');axes[0].tick_params(axis='x',rotation=15)
    axes[1].bar(table.method,table.mean_total_tokens)
    axes[1].set_title('Mean input + output tokens');axes[1].tick_params(axis='x',rotation=15)
    fig.tight_layout();fig.savefig(directory/'comparison.png',dpi=160);plt.show()


## 6. Download results or a checkpoint
Run after completing or stopping the benchmark. The ZIP includes code, exact configuration, dataset provenance, selected questions and answer keys, per-problem results, summaries, raw calls, traces and plots. It excludes model weights. Also save Kaggle notebook outputs.

For another session, attach your own ZIP as input and extract it under `/kaggle/working` before running setup; it restores the `crosswords10_schema_v3` directory. Resume requires the same experiment/runtime. A GPU/session interruption before a call is saved leaves unknown cost; use a fresh trial for clean cost measurement.


In [ ]:
from IPython.display import FileLink
export=Path('/kaggle/working')/f'crosswords_schema_v3_n{len(puzzles)}_exp{config["max_expansions"]}_results.zip'
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.name!='run.lock':
            archive.write(path,Path('crosswords10_schema_v3')/path.relative_to(PROJECT))
previous_dir=Path.cwd()
try:
    os.chdir(export.parent)
    display(FileLink(export.name))
finally:
    os.chdir(previous_dir)
